## Task 1

In [4]:
import random
from collections import deque

# BFS
def shortest_path(env, start, goal):
    q = deque([(start, 0)])
    visited = {start}

    while q:
        pos, dist = q.popleft()
        if pos == goal:
            return dist

        for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
            nxt = (pos[0] + dr, pos[1] + dc)
            if not env.is_blocked(nxt) and nxt not in visited:
                visited.add(nxt)
                q.append((nxt, dist + 1))
    return None


# Environment
class GridEnvironment:
    def __init__(self, size=5, obstacle_count=4):
        self.size = size
        self.grid = [["." for _ in range(size)] for _ in range(size)]
        self.goal = (size - 1, size - 1)
        self.obstacles = []

        count = 0
        while count < obstacle_count:
            r, c = random.randint(0, size-1), random.randint(0, size-1)
            if (r, c) != (0, 0) and (r, c) != self.goal and (r, c) not in self.obstacles:
                self.grid[r][c] = "O"
                self.obstacles.append((r, c))
                count += 1

    def is_blocked(self, pos):
        r, c = pos
        return not (0 <= r < self.size and 0 <= c < self.size) or self.grid[r][c] == "O"

    def print_grid(self):
        print("\nGrid:")
        for r in range(self.size):
            row = []
            for c in range(self.size):
                if (r, c) == (0, 0):
                    row.append("S")
                elif (r, c) == self.goal:
                    row.append("G")
                else:
                    row.append(self.grid[r][c])
            print(" ".join(row))


# Simple Reflex Agent
class SimpleReflexAgent:
    def __init__(self):
        self.name = "Simple Reflex"

    def act(self, current_pos, env):
        r, c = current_pos
        directions = {
            "DOWN": (r + 1, c),
            "RIGHT": (r, c + 1),
            "UP": (r - 1, c),
            "LEFT": (r, c - 1)
        }

        for action, next_pos in directions.items():
            if next_pos == env.goal:
                return action

        valid_actions = [a for a, p in directions.items() if not env.is_blocked(p)]
        return random.choice(valid_actions) if valid_actions else None


# Model-Based Agent
class ModelBasedAgent:
    def __init__(self):
        self.name = "Model-Based"
        self.visited = set()

    def act(self, current_pos, env):
        self.visited.add(current_pos)
        r, c = current_pos

        directions = {
            "DOWN": (r + 1, c),
            "RIGHT": (r, c + 1),
            "UP": (r - 1, c),
            "LEFT": (r, c - 1)
        }

        for action, next_pos in directions.items():
            if next_pos == env.goal:
                return action

        unvisited = [a for a, p in directions.items()
                     if not env.is_blocked(p) and p not in self.visited]

        if unvisited:
            return random.choice(unvisited)

        valid = [a for a, p in directions.items()
                 if not env.is_blocked(p)]

        return random.choice(valid) if valid else None


# Test Case Runner
def run_test_case(test_id):
    print(f"\n===== Test Case {test_id} =====")

    env = GridEnvironment()
    env.print_grid()

    start_pos = (0, 0)
    optimal_len = shortest_path(env, start_pos, env.goal)

    for AgentClass in [SimpleReflexAgent, ModelBasedAgent]:
        agent = AgentClass()
        pos = start_pos
        steps = 0
        path = [pos]
        actions_taken = []
        reached = False
        good_moves = 0

        while steps < 30:
            if pos == env.goal:
                reached = True
                break

            old_dist = abs(pos[0] - env.goal[0]) + abs(pos[1] - env.goal[1])

            action = agent.act(pos, env)
            if not action:
                break

            actions_taken.append(action)

            dr, dc = {
                "UP": (-1, 0),
                "DOWN": (1, 0),
                "LEFT": (0, -1),
                "RIGHT": (0, 1)
            }[action]

            pos = (pos[0] + dr, pos[1] + dc)
            path.append(pos)
            steps += 1

            new_dist = abs(pos[0] - env.goal[0]) + abs(pos[1] - env.goal[1])
            if new_dist < old_dist:
                good_moves += 1

        revisits = len(path) - len(set(path))
        deviation = None if optimal_len is None else steps - optimal_len

        print(f"\nAgent: {agent.name}")
        print("Path taken:", path)
        print("Actions taken:", actions_taken)
        print("Goal reached:", "Yes" if reached else "No")
        print("Total steps:", steps)
        print("Loops:", revisits)
        print("Optimal deviation:", deviation)
        print("Good moves:", good_moves)


print("Performance Comparison (5 Test Cases):")

for i in range(1, 6):
    run_test_case(i)


Performance Comparison (5 Test Cases):

===== Test Case 1 =====

Grid:
S O . O .
. . O . .
. . . . .
. . . O .
. . . . G

Agent: Simple Reflex
Path taken: [(0, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Actions taken: ['DOWN', 'DOWN', 'UP', 'DOWN', 'UP', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'LEFT', 'RIGHT', 'DOWN', 'RIGHT', 'RIGHT']
Goal reached: Yes
Total steps: 14
Loops: 6
Optimal deviation: 6
Good moves: 11

Agent: Model-Based
Path taken: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Actions taken: ['DOWN', 'RIGHT', 'DOWN', 'LEFT', 'DOWN', 'DOWN', 'RIGHT', 'UP', 'RIGHT', 'DOWN', 'RIGHT', 'RIGHT']
Goal reached: Yes
Total steps: 12
Loops: 0
Optimal deviation: 4
Good moves: 10

===== Test Case 2 =====

Grid:
S . . . .
. . O O .
. . . . .
. . . . O
. O . . G

Agent: Simple Reflex
Path taken: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 0), (2, 0), (3, 0), (3, 

## Task 2

In [3]:
import random

GRID_SIZE = 5
OBSTACLE_COUNT = 5
STEP_LIMIT = 30
TEST_CASES = 5

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

# Environment
class SmartCampus:
    def __init__(self, size=GRID_SIZE):
        self.size = size
        self.grid = [["." for _ in range(size)] for _ in range(size)]
        self.start = (0, 0)
        self.goal = (size - 1, size - 1)
        self.grid[0][0] = "S"
        self.grid[size - 1][size - 1] = "G"
        self.place_obstacles()

    def place_obstacles(self):
        count = 0
        while count < OBSTACLE_COUNT:
            r = random.randint(0, self.size - 1)
            c = random.randint(0, self.size - 1)
            if (r, c) not in [self.start, self.goal] and self.grid[r][c] == ".":
                self.grid[r][c] = "O"
                count += 1

    def is_blocked(self, pos):
        r, c = pos
        return not (0 <= r < self.size and 0 <= c < self.size) or self.grid[r][c] == "O"

    def risk_level(self, pos):
        r, c = pos
        risk = 0
        for dr, dc in ACTIONS.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.size and 0 <= nc < self.size:
                if self.grid[nr][nc] == "O":
                    risk += 1
        return risk

    def print_grid(self):
        print("\nGrid:")
        for row in self.grid:
            print(" ".join(row))


# Goal-Based Agent
class GoalBasedAgent:
    name = "Goal-Based"
    def act(self, pos, env):
        r, c = pos
        gr, gc = env.goal
        best_action = None
        best_dist = float("inf")
        for act, (dr, dc) in ACTIONS.items():
            nxt = (r + dr, c + dc)
            if env.is_blocked(nxt):
                continue
            dist = abs(nxt[0] - gr) + abs(nxt[1] - gc)
            if dist < best_dist:
                best_dist = dist
                best_action = act
        return best_action


# Utility-Based Agent
class UtilityAgent:
    name = "Utility-Based"
    W_DIST = -3
    W_ENERGY = -1
    W_RISK = -2
    def act(self, pos, env):
        r, c = pos
        gr, gc = env.goal
        best_action = None
        best_utility = -float("inf")
        for act, (dr, dc) in ACTIONS.items():
            nxt = (r + dr, c + dc)
            if env.is_blocked(nxt):
                continue
            dist = abs(nxt[0] - gr) + abs(nxt[1] - gc)
            energy_cost = 1
            risk = env.risk_level(nxt)
            utility = self.W_DIST * dist + self.W_ENERGY * energy_cost + self.W_RISK * risk
            if utility > best_utility:
                best_utility = utility
                best_action = act
        return best_action


# Agent runner
def run_agent(agent, env):
    pos = env.start
    steps = 0
    path = [pos]
    actions_taken = []
    reached = False
    while steps < STEP_LIMIT:
        if pos == env.goal:
            reached = True
            break
        action = agent.act(pos, env)
        if action is None:
            break
        actions_taken.append(action)
        dr, dc = ACTIONS[action]
        pos = (pos[0] + dr, pos[1] + dc)
        path.append(pos)
        steps += 1
    revisits = len(path) - len(set(path))
    return {
        "steps": steps,
        "reached": reached,
        "loops": revisits,
        "path": path,
        "actions": actions_taken
    }


print("\nGoal-Based vs Utility-Based Comparison")

goal_success = {"Goal-Based": 0, "Utility-Based": 0}

for case in range(1, TEST_CASES + 1):
    print("\n======================")
    print("Test Case", case)
    print("======================")

    env = SmartCampus()
    env.print_grid()

    goal_agent = GoalBasedAgent()
    utility_agent = UtilityAgent()

    goal_res = run_agent(goal_agent, env)
    util_res = run_agent(utility_agent, env)

    if goal_res["reached"]:
        goal_success["Goal-Based"] += 1
    if util_res["reached"]:
        goal_success["Utility-Based"] += 1

    # Only print summary per agent
    for agent_name, res in [("Goal-Based", goal_res), ("Utility-Based", util_res)]:
        print(f"\n{agent_name} Agent Summary:")
        print("Goal reached:", "Yes" if res["reached"] else "No")
        print("Total steps:", res["steps"])
        print("Loops (revisited cells):", res["loops"])
        print("Path:", res["path"])
        print("Actions:", res["actions"])

print("\nGoal Success Rate:")
for k, v in goal_success.items():
    print(f"{k}: {v}/{TEST_CASES}")



Goal-Based vs Utility-Based Comparison

Test Case 1

Grid:
S . . . .
O . . . .
O . O . .
O . O . .
. . . . G

Goal-Based Agent Summary:
Goal reached: Yes
Total steps: 8
Loops (revisited cells): 0
Path: [(0, 0), (0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]
Actions: ['RIGHT', 'DOWN', 'DOWN', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'RIGHT']

Utility-Based Agent Summary:
Goal reached: Yes
Total steps: 8
Loops (revisited cells): 0
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (3, 4), (4, 4)]
Actions: ['RIGHT', 'RIGHT', 'RIGHT', 'DOWN', 'RIGHT', 'DOWN', 'DOWN', 'DOWN']

Test Case 2

Grid:
S . . . .
. . . . .
. O O . .
O O . . .
. . O . G

Goal-Based Agent Summary:
Goal reached: No
Total steps: 30
Loops (revisited cells): 28
Path: [(0, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1, 0), (2, 0), (1